# Whisper SER Finetuning
This notebook fine-tunes `openai/whisper-large-v3` for speech emotion recognition on IEMOCAP and can train two variants: one **with `xxx`** and one **without `xxx`**. By default, both are produced so you can compare the effect directly.


## 1) Imports and Notebook Settings
This cell loads the training dependencies and prints the runtime info before we touch the dataset or model.


In [1]:
import accelerate, transformers
print(accelerate.__version__)
print(transformers.__version__)


1.13.0
5.1.0


In [31]:
from pathlib import Path
import json

import librosa
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    Trainer,
    TrainingArguments,
    default_data_collator,
)

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
np.random.seed(42)

print('Imports loaded.')
print(f'Torch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')


Imports loaded.
Torch version: 2.10.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 5080


## 2) Configuration
`INCLUDE_XXX` controls the single-run behavior, while `RUN_BOTH_XXX_VARIANTS` lets the notebook train both setups and compare them in one place.


In [32]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root from the notebook location.')


if 'whisper' in str(Path.cwd()).replace('\\', '/').lower():
    REPO_ROOT = find_repo_root(Path.cwd().resolve())
else:
    REPO_ROOT = Path.cwd().resolve()

MODEL_ID = 'openai/whisper-large-v3'
BASE_TARGET_LABELS = ['ang', 'fea', 'hap', 'neu', 'sad', 'sur']
TEST_SIZE = 0.20
VAL_SIZE = 0.10
RANDOM_STATE = 42
MAX_DURATION = 30.0
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None
LEARNING_RATE = 5e-5
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
NUM_TRAIN_EPOCHS = 5.0
WARMUP_RATIO = 0.10
LOGGING_STEPS = 10
FREEZE_ENCODER = False
USE_FP16 = False
RUN_BOTH_XXX_VARIANTS = True
INCLUDE_XXX = True
OUTPUT_DIR = REPO_ROOT / 'whisper' / 'finetuning' / 'artifacts' / 'whisper_large_v3_iemocap_variants'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_IEMOCAP_TO_LABEL = {
    'ang': 'ang',
    'fea': 'fea',
    'hap': 'hap',
    'neu': 'neu',
    'sad': 'sad',
    'sur': 'sur',
}


def resolve_precision_flags(request_fp16: bool):
    has_cuda = torch.cuda.is_available()
    use_fp16 = bool(request_fp16 and has_cuda)
    use_bf16 = False
    return use_fp16, use_bf16


EFFECTIVE_FP16, EFFECTIVE_BF16 = resolve_precision_flags(USE_FP16)

config_df = pd.DataFrame([
    ('repo_root', str(REPO_ROOT)),
    ('model_id', MODEL_ID),
    ('base_target_labels', ', '.join(BASE_TARGET_LABELS)),
    ('test_size', TEST_SIZE),
    ('val_size', VAL_SIZE),
    ('random_state', RANDOM_STATE),
    ('max_duration', MAX_DURATION),
    ('max_train_samples', MAX_TRAIN_SAMPLES),
    ('max_eval_samples', MAX_EVAL_SAMPLES),
    ('learning_rate', LEARNING_RATE),
    ('train_batch_size', TRAIN_BATCH_SIZE),
    ('eval_batch_size', EVAL_BATCH_SIZE),
    ('gradient_accumulation_steps', GRADIENT_ACCUMULATION_STEPS),
    ('num_train_epochs', NUM_TRAIN_EPOCHS),
    ('freeze_encoder', FREEZE_ENCODER),
    ('fp16_requested', USE_FP16),
    ('fp16_effective', EFFECTIVE_FP16),
    ('bf16_effective', EFFECTIVE_BF16),
    ('include_xxx', INCLUDE_XXX),
    ('run_both_xxx_variants', RUN_BOTH_XXX_VARIANTS),
    ('output_dir', str(OUTPUT_DIR)),
], columns=['setting', 'value'])

display(config_df)


,setting,value
0,repo_root,/mnt/f/Speech-Emotion-Recognition
1,model_id,openai/whisper-large-v3
2,base_target_labels,"ang, fea, hap, neu, sad, sur"
3,test_size,0.2
4,val_size,0.1
5,random_state,42
6,max_duration,30.0
7,max_train_samples,None
8,max_eval_samples,None
9,learning_rate,0.00005


## 3) Variant Helpers
These helpers define the label policy for the `with_xxx` and `without_xxx` runs.


In [33]:
def target_labels_for_variant(include_xxx: bool) -> list[str]:
    return [*BASE_TARGET_LABELS, 'xxx'] if include_xxx else list(BASE_TARGET_LABELS)


def label_map_for_variant(include_xxx: bool) -> dict[str, str]:
    label_map = dict(BASE_IEMOCAP_TO_LABEL)
    if include_xxx:
        label_map['xxx'] = 'xxx'
    return label_map


def variant_name(include_xxx: bool) -> str:
    return 'with_xxx' if include_xxx else 'without_xxx'


variants_to_run = [True, False] if RUN_BOTH_XXX_VARIANTS else [INCLUDE_XXX]
variant_preview_df = pd.DataFrame([
    {
        'variant': variant_name(include_xxx),
        'include_xxx': include_xxx,
        'target_labels': ', '.join(target_labels_for_variant(include_xxx)),
    }
    for include_xxx in variants_to_run
])

display(variant_preview_df)


,variant,include_xxx,target_labels
0,with_xxx,True,"ang, fea, hap, neu, sad, sur, xxx"
1,without_xxx,False,"ang, fea, hap, neu, sad, sur"


## 4) Load Metadata and Preview Both Label Policies
This cell shows how many rows and classes each finetuning variant will use before we actually start training.


In [34]:
METADATA_PATH = REPO_ROOT / 'datasets' / 'IEMOCAP' / 'iemocap_full_dataset.csv'
AUDIO_ROOT = REPO_ROOT / 'datasets' / 'IEMOCAP'

raw_df = pd.read_csv(METADATA_PATH)
raw_df['emotion'] = raw_df['emotion'].astype(str).str.strip().str.lower()
raw_df['method'] = raw_df['method'].astype(str).str.strip().str.lower()
raw_df['gender'] = raw_df['gender'].astype(str).str.strip().str.upper()
raw_df['path'] = raw_df['path'].astype(str).str.replace('\\', '/', regex=False).str.strip()


def build_training_frame(include_xxx: bool) -> pd.DataFrame:
    label_map = label_map_for_variant(include_xxx)
    df = raw_df[raw_df['emotion'].isin(label_map)].copy()
    if include_xxx:
        df = df[(df['emotion'] == 'xxx') | (df['agreement'] > 0)].copy()
    else:
        df = df[df['agreement'] > 0].copy()
    df['label'] = df['emotion'].map(label_map)
    df['audio_path'] = df['path'].map(lambda rel: str((AUDIO_ROOT / rel).resolve()))
    df['audio_exists'] = df['audio_path'].map(lambda p: Path(p).exists())
    df = df[df['audio_exists']].reset_index(drop=True)
    label_to_id = {label: idx for idx, label in enumerate(target_labels_for_variant(include_xxx))}
    df['label_id'] = df['label'].map(label_to_id)
    return df


variant_frames = {variant_name(include_xxx): build_training_frame(include_xxx) for include_xxx in variants_to_run}
summary_rows = []
for include_xxx in variants_to_run:
    name = variant_name(include_xxx)
    frame = variant_frames[name]
    summary_rows.append({
        'variant': name,
        'include_xxx': include_xxx,
        'rows': len(frame),
        'num_labels': len(target_labels_for_variant(include_xxx)),
        'labels': ', '.join(target_labels_for_variant(include_xxx)),
    })

display(pd.DataFrame(summary_rows))
for name, frame in variant_frames.items():
    print(f'Label distribution for {name}:')
    display(frame['label'].value_counts().rename_axis('label').reset_index(name='count'))


,variant,include_xxx,rows,num_labels,labels
0,with_xxx,True,7144,7,"ang, fea, hap, neu, sad, sur, xxx"
1,without_xxx,False,4637,6,"ang, fea, hap, neu, sad, sur"


Label distribution for with_xxx:


,label,count
0,xxx,2507
1,neu,1708
2,ang,1103
3,sad,1084
4,hap,595
5,sur,107
6,fea,40


Label distribution for without_xxx:


,label,count
0,neu,1708
1,ang,1103
2,sad,1084
3,hap,595
4,sur,107
5,fea,40


## 5) Split Helpers and Audio Dataset Wrapper
This defines the stratified train/validation/test split logic, the audio loader, and the dataset wrapper used by Hugging Face `Trainer`.


In [35]:
def make_splits(df: pd.DataFrame):
    train_val_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=df['label'],
    )
    relative_val_size = VAL_SIZE / (1.0 - TEST_SIZE)
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=relative_val_size,
        random_state=RANDOM_STATE,
        stratify=train_val_df['label'],
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


def load_audio(audio_path: Path, target_sr: int, max_duration: float) -> np.ndarray:
    audio, _ = librosa.load(audio_path, sr=target_sr, mono=True)
    max_length = int(target_sr * max_duration)
    if len(audio) >= max_length:
        return audio[:max_length].astype(np.float32, copy=False)
    padded = np.zeros(max_length, dtype=np.float32)
    padded[: len(audio)] = audio.astype(np.float32, copy=False)
    return padded


class IemocapWhisperDataset(torch.utils.data.Dataset):
    def __init__(self, frame: pd.DataFrame, feature_extractor, max_duration: float):
        self.frame = frame.reset_index(drop=True)
        self.feature_extractor = feature_extractor
        self.max_duration = max_duration

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        audio = load_audio(
            Path(row['audio_path']),
            target_sr=self.feature_extractor.sampling_rate,
            max_duration=self.max_duration,
        )
        features = self.feature_extractor(
            audio,
            sampling_rate=self.feature_extractor.sampling_rate,
            truncation=True,
            max_length=int(self.feature_extractor.sampling_rate * self.max_duration),
            return_tensors='pt',
        )
        item = {key: value.squeeze(0) for key, value in features.items()}
        item['labels'] = torch.tensor(int(row['label_id']), dtype=torch.long)
        return item


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'precision_weighted': float(precision_score(labels, preds, average='weighted', zero_division=0)),
        'recall_weighted': float(recall_score(labels, preds, average='weighted', zero_division=0)),
        'f1_weighted': float(f1_score(labels, preds, average='weighted', zero_division=0)),
        'f1_macro': float(f1_score(labels, preds, average='macro', zero_division=0)),
    }


## 6) Load the Whisper Feature Extractor Once
We reuse the same feature extractor for both variants.


In [36]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID, do_normalize=True)
print(f'Feature extractor sample rate: {feature_extractor.sampling_rate}')


Feature extractor sample rate: 16000


## 7) Prepare Variant Datasets and Split Summaries
This cell creates the train/validation/test splits for each variant and shows the split sizes before training starts.


In [37]:
variant_splits = {}
variant_split_summary_rows = []

for include_xxx in variants_to_run:
    name = variant_name(include_xxx)
    frame = variant_frames[name]
    train_df, val_df, test_df = make_splits(frame)

    if MAX_TRAIN_SAMPLES is not None:
        train_df = train_df.iloc[:MAX_TRAIN_SAMPLES].reset_index(drop=True)
    if MAX_EVAL_SAMPLES is not None:
        val_df = val_df.iloc[:MAX_EVAL_SAMPLES].reset_index(drop=True)
        test_df = test_df.iloc[:MAX_EVAL_SAMPLES].reset_index(drop=True)

    variant_splits[name] = {
        'include_xxx': include_xxx,
        'target_labels': target_labels_for_variant(include_xxx),
        'train_df': train_df,
        'val_df': val_df,
        'test_df': test_df,
    }

    variant_split_summary_rows.extend([
        {'variant': name, 'split': 'train', 'rows': len(train_df)},
        {'variant': name, 'split': 'val', 'rows': len(val_df)},
        {'variant': name, 'split': 'test', 'rows': len(test_df)},
    ])

variant_split_summary_df = pd.DataFrame(variant_split_summary_rows)
display(variant_split_summary_df)


,variant,split,rows
0,with_xxx,train,5000
1,with_xxx,val,715
2,with_xxx,test,1429
3,without_xxx,train,3245
4,without_xxx,val,464
5,without_xxx,test,928


## 8) Define a Single-Variant Training Function
This function fine-tunes one Whisper variant and returns a compact summary so we can compare `with_xxx` and `without_xxx` afterwards.


In [38]:
def train_variant(include_xxx: bool) -> dict:
    name = variant_name(include_xxx)
    target_labels = target_labels_for_variant(include_xxx)
    label_to_id = {label: idx for idx, label in enumerate(target_labels)}
    id_to_label = {idx: label for label, idx in label_to_id.items()}
    variant_out_dir = OUTPUT_DIR / name
    variant_out_dir.mkdir(parents=True, exist_ok=True)
    use_fp16, use_bf16 = resolve_precision_flags(USE_FP16)
    split_bundle = variant_splits[name]
    train_df = split_bundle['train_df']
    val_df = split_bundle['val_df']
    test_df = split_bundle['test_df']
    model = AutoModelForAudioClassification.from_pretrained(
        MODEL_ID,
        num_labels=len(target_labels),
        label2id=label_to_id,
        id2label=id_to_label,
        ignore_mismatched_sizes=True,
    )
    if not use_fp16 and not use_bf16:
        model = model.float()
    if FREEZE_ENCODER and hasattr(model, 'freeze_encoder'):
        model.freeze_encoder()
    train_dataset = IemocapWhisperDataset(train_df, feature_extractor=feature_extractor, max_duration=MAX_DURATION)
    val_dataset = IemocapWhisperDataset(val_df, feature_extractor=feature_extractor, max_duration=MAX_DURATION)
    test_dataset = IemocapWhisperDataset(test_df, feature_extractor=feature_extractor, max_duration=MAX_DURATION)
    training_args = TrainingArguments(
        output_dir=str(variant_out_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=LOGGING_STEPS,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        greater_is_better=True,
        report_to='none',
        fp16=use_fp16,
        bf16=use_bf16,
        remove_unused_columns=False,
        save_total_limit=2,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=default_data_collator,
        compute_metrics=compute_metrics,
    )
    train_result = trainer.train()
    val_metrics = trainer.evaluate(eval_dataset=val_dataset, metric_key_prefix='val')
    test_metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix='test')
    trainer.save_model()
    feature_extractor.save_pretrained(variant_out_dir)
    split_summary_df = pd.DataFrame([
        {'split': 'train', 'rows': len(train_df)},
        {'split': 'val', 'rows': len(val_df)},
        {'split': 'test', 'rows': len(test_df)},
    ])
    split_summary_df.to_csv(variant_out_dir / 'split_summary.csv', index=False)
    metadata = {
        'variant': name,
        'include_xxx': include_xxx,
        'target_labels': target_labels,
        'fp16_requested': USE_FP16,
        'fp16_effective': use_fp16,
        'bf16_effective': use_bf16,
        'train_metrics': train_result.metrics,
        'val_metrics': val_metrics,
        'test_metrics': test_metrics,
    }
    (variant_out_dir / 'run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    summary = {
        'variant': name,
        'include_xxx': include_xxx,
        'target_labels': ', '.join(target_labels),
        'fp16_requested': USE_FP16,
        'fp16_effective': use_fp16,
        'bf16_effective': use_bf16,
        'train_rows': len(train_df),
        'val_rows': len(val_df),
        'test_rows': len(test_df),
        'output_dir': str(variant_out_dir),
    }
    for key, value in test_metrics.items():
        if isinstance(value, (int, float)):
            summary[key] = float(value)
    return summary


## 9) Train the Requested Variant(s)
Run this cell to fine-tune the model. By default it trains both `with_xxx` and `without_xxx` and collects a comparison table.


In [39]:
variant_results = [train_variant(include_xxx) for include_xxx in variants_to_run]
variant_results_df = pd.DataFrame(variant_results)
display(variant_results_df)


Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

WhisperForAudioClassification LOAD REPORT from: openai/whisper-large-v3
Key                                                          | Status     | 
-------------------------------------------------------------+------------+-
model.decoder.layers.{0...31}.self_attn.out_proj.bias        | UNEXPECTED | 
model.decoder.layers.{0...31}.self_attn.q_proj.bias          | UNEXPECTED | 
model.decoder.layers.{0...31}.fc2.weight                     | UNEXPECTED | 
model.decoder.layers.{0...31}.self_attn.v_proj.bias          | UNEXPECTED | 
model.decoder.layers.{0...31}.encoder_attn.out_proj.weight   | UNEXPECTED | 
model.decoder.layers.{0...31}.encoder_attn.v_proj.bias       | UNEXPECTED | 
model.decoder.layers.{0...31}.self_attn.q_proj.weight        | UNEXPECTED | 
model.decoder.layers.{0...31}.final_layer_norm.weight        | UNEXPECTED | 
model.decoder.layers.{0...31}.encoder_attn.q_proj.weight     | UNEXPECTED | 
model.decoder.layers.{0...31}.encoder_attn_layer_norm.bias   | UNEXPECTED | 
mode

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 10) Save Variant Comparison Summary
This saves a compact comparison table at the base finetuning artifact folder so you can see both runs side by side.


In [ ]:
variant_results_df.to_csv(OUTPUT_DIR / 'variant_comparison.csv', index=False)
(OUTPUT_DIR / 'variant_comparison.json').write_text(
    json.dumps(variant_results, indent=2),
    encoding='utf-8',
)

artifact_df = pd.DataFrame([
    ('variant_comparison_csv', str(OUTPUT_DIR / 'variant_comparison.csv')),
    ('variant_comparison_json', str(OUTPUT_DIR / 'variant_comparison.json')),
    *[(f'{row.variant}_dir', row.output_dir) for row in variant_results_df.itertuples(index=False)],
], columns=['artifact', 'path'])

display(artifact_df)


## 11) Notes
- `xxx` is included by default in single-run mode.
- By default, this notebook trains both variants so you get results with and without `xxx`.
- The `with_xxx` run keeps `xxx` even when annotator agreement is not positive.
- `exc`, `fru`, `oth`, and `dis` are still excluded from finetuning.
- For a smoke run, set `MAX_TRAIN_SAMPLES` and `MAX_EVAL_SAMPLES` in the config cell.
